# Fourier-space KNN → MAF-joint posterior evaluation

**Approach**: for each real patient, find the nearest sim in Fourier space
(FFT magnitude of 4 pressure channels, first 30 coefficients, unweighted L2).
Then sample the MAF-joint posterior conditioned on that sim's waveform.

**Posterior**: `exp_cnn4e64-ae-reduced_maf5_freeze-maf_1M` (best reduced-input model, 1M sims).

**Caches** (from `sim_real_gap.ipynb`):
- `outputs/x_sim_raw_1000000.pt` — 1M sim input vectors (ReducedCVDataset format, 809-dim)
- `outputs/f_sim_fft30_1000000.pt` — 1M Fourier feature vectors (4×30=120-dim)
- `outputs/exp_cnn4e64-ae-reduced_maf5_freeze-maf_1M/theta_sim_1000000.pt` — 1M sim parameters

**Sections**:
1. KNN search in Fourier space (all 1M sims, GPU one-shot cdist)
2. Distance diagnostics vs sim-sim baseline
3. Waveform overlay: best / worst by Fourier distance
4. MAF-joint posterior on nearest sim waveforms
5. GT scatter — Cas, Eap, Rap (PVR), Ras (SVR)
6. MAPE vs NN distance
7. Nearest-sim parameter ceiling check
8. Parameter distributions (posterior histograms per patient)
9. Comparison table: Fourier-NN vs latent-space NN

In [ ]:
import sys, h5py, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

ROOT = Path(globals()['_dh'][0])
sys.path.insert(0, str(ROOT))

from dataset import load_stats, load_manifest, ReducedCVDataset, WAVE_KEYS_REDUCED, PARAM_KEYS_INFER

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

BASE_RUN   = 'exp_cnn4e64-ae-reduced_maf5_freeze-maf_1M'
SIM_ROOT   = Path('/media/local/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314')
REAL_DATA  = Path('/home/sa4604/real_data/multibeat')
STATS_PATH = ROOT / 'norm_stats.json'

N_SIM   = 1_000_000
N_FREQ  = 30
T       = 201
N_POSTERIOR_SAMPLES = 1000

CACHE_RAW   = ROOT / f'outputs/x_sim_raw_{N_SIM}.pt'
CACHE_FFT   = ROOT / f'outputs/f_sim_fft{N_FREQ}_{N_SIM}.pt'
CACHE_THETA = ROOT / f'outputs/{BASE_RUN}/theta_sim_{N_SIM}.pt'

stats    = load_stats(STATS_PATH)
manifest = load_manifest(SIM_ROOT / 'manifest_train.json')
print(f'Caches:\n  raw:   {CACHE_RAW}\n  fft:   {CACHE_FFT}\n  theta: {CACHE_THETA}')

## Load MAF-joint posterior

In [ ]:
posterior = torch.load(
    ROOT / f'outputs/{BASE_RUN}/posterior.pt', map_location=device, weights_only=False
)
print('Posterior loaded:', type(posterior).__name__)

lo_t = torch.tensor([manifest['config']['pvar_low'][k]  for k in PARAM_KEYS_INFER], dtype=torch.float32)
hi_t = torch.tensor([manifest['config']['pvar_high'][k] for k in PARAM_KEYS_INFER], dtype=torch.float32)
print(f'Prior bounds loaded: {len(PARAM_KEYS_INFER)} params')

## Load real patients

In [ ]:
pas_mean = stats['waves']['Pas']['mean'];  pas_std = stats['waves']['Pas']['std'] + 1e-8
vlv_std  = stats['waves']['Vlv']['std']  + 1e-8
hr_mean  = stats['parameters']['HR']['mean']; hr_std = stats['parameters']['HR']['std'] + 1e-8
wm = torch.tensor([stats['waves'][k]['mean'] for k in WAVE_KEYS_REDUCED], dtype=torch.float32).unsqueeze(1)
ws = torch.tensor([stats['waves'][k]['std']  for k in WAVE_KEYS_REDUCED], dtype=torch.float32).unsqueeze(1) + 1e-8

wave_mean_np = wm.squeeze().numpy()
wave_std_np  = ws.squeeze().numpy()

patients = []
for fpath in sorted(REAL_DATA.glob('*.h5')):
    beats_x, gt = [], {}
    with h5py.File(fpath, 'r') as f:
        for bk in sorted(f.keys()):
            if not bk.startswith('beat_'): continue
            g = f[bk]
            waves = np.stack([g[f'waves/{k}'][:].astype(np.float32) for k in WAVE_KEYS_REDUCED])
            waves_t = (torch.from_numpy(waves) - wm) / ws
            sbp  = float(g['summaries/sbp'][()])
            dbp  = float(g['summaries/dbp'][()])
            map_ = float(g['summaries/map'][()])
            sv   = float(g['summaries/sv'][()])
            hr   = float(g['parameters/HR'][()])
            scalars = torch.tensor([
                (map_ - pas_mean) / pas_std, (sbp - pas_mean) / pas_std,
                (dbp  - pas_mean) / pas_std, sv / vlv_std,
                (hr   - hr_mean)  / hr_std,
            ], dtype=torch.float32)
            beats_x.append(torch.cat([waves_t.reshape(-1), scalars]))
            if not gt:
                gt = {k: float(g[f'parameters/{k}'][()]) for k in f[bk]['parameters'].keys()}
                gt['sbp'] = sbp; gt['dbp'] = dbp; gt['map'] = map_; gt['sv'] = sv
    if beats_x:
        x_beats = torch.stack(beats_x)
        raw_waves = x_beats.mean(dim=0)[:4*T].view(4, T).numpy() * wave_std_np[:, None] + wave_mean_np[:, None]
        patients.append(dict(file=fpath.stem, x_avg=x_beats.mean(dim=0), raw_waves=raw_waves, gt=gt))

x_real = torch.stack([p['x_avg'] for p in patients])   # (60, 809)
print(f'Loaded {len(patients)} patients  x_real: {tuple(x_real.shape)}')

## Fourier features + KNN search (all 1M sims)

FFT magnitude of each of the 4 pressure waveform channels, first `N_FREQ` coefficients.
Unweighted L2 in the 120-dim feature space — same as `sim_real_gap.ipynb`.

In [ ]:
def fourier_features(x: torch.Tensor, n_freq: int = N_FREQ) -> torch.Tensor:
    """FFT magnitude of the 4 reduced pressure channels. x: (..., 809) → (..., 4*n_freq)"""
    waves = x[..., :4 * T].reshape(*x.shape[:-1], 4, T)
    mag   = torch.fft.rfft(waves, dim=-1).abs()[..., :n_freq]
    return mag.reshape(*x.shape[:-1], 4 * n_freq)

print('Loading cached sim features...')
t0 = time.time()
x_sim = torch.load(CACHE_RAW, map_location='cpu')
f_sim = torch.load(CACHE_FFT, map_location='cpu')
print(f'Loaded in {time.time()-t0:.1f}s  x_sim: {tuple(x_sim.shape)}  f_sim: {tuple(f_sim.shape)}')

# Fourier features for real patients (same transform as sims)
f_real = fourier_features(x_real)   # (60, 4*N_FREQ)

print(f'Moving f_sim to GPU ({f_sim.nbytes/1e9:.2f} GB)...')
f_sim_gpu  = f_sim.to(device)
f_real_gpu = f_real.to(device)

print('Computing NN in Fourier space (60 × 1M cdist)...')
t0 = time.time()
dists_fft = torch.cdist(f_real_gpu, f_sim_gpu)   # (60, 1M)
nn_dist_fft, nn_idx_fft = dists_fft.min(dim=1)
nn_dist_fft = nn_dist_fft.cpu()
nn_idx_fft  = nn_idx_fft.cpu()
del dists_fft
print(f'Done in {time.time()-t0:.1f}s')

# Sim-sim baseline
rand_i      = torch.randperm(N_SIM)[:200]
sim_sim_fft = torch.cdist(f_sim_gpu[rand_i[:100]], f_sim_gpu[rand_i[100:]]).flatten().cpu()
del f_sim_gpu
torch.cuda.empty_cache()

print(f'\nFourier space ({4*N_FREQ}-dim L2):')
print(f'  sim-sim random : mean={sim_sim_fft.mean():.3f}  median={sim_sim_fft.median():.3f}')
print(f'  real → NN sim  : mean={nn_dist_fft.mean():.3f}  median={nn_dist_fft.median():.3f}  '
      f'min={nn_dist_fft.min():.3f}  max={nn_dist_fft.max():.3f}')
print(f'  ratio          : {(nn_dist_fft.mean() / sim_sim_fft.mean()).item():.2f}x sim-sim')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(nn_dist_fft.numpy(), bins=20, color='mediumseagreen', alpha=0.8, label='real → NN dist')
ax.axvline(sim_sim_fft.median().item(), color='steelblue', linestyle='--', linewidth=1.5,
           label=f'sim-sim median ({sim_sim_fft.median():.2f})')
ax.set_xlabel(f'L2 distance in Fourier space ({4*N_FREQ}-dim)')
ax.set_title('Fourier NN distances: real → nearest sim')
ax.legend(); plt.tight_layout(); plt.show()

## Waveform overlay — best and worst matches

In [ ]:
def plot_overlay(patient_indices, nn_indices, title, dist_arr):
    n = len(patient_indices)
    fig, axes = plt.subplots(n, 4, figsize=(18, 3 * n))
    if n == 1: axes = axes[None, :]
    for row, (pi, ni) in enumerate(zip(patient_indices, nn_indices)):
        real_phys = patients[pi]['raw_waves']                          # (4, T) physical
        sim_norm  = x_sim[ni, :4*T].view(4, T).numpy()                # z-scored
        sim_phys  = sim_norm * wave_std_np[:, None] + wave_mean_np[:, None]
        for ci, ch in enumerate(WAVE_KEYS_REDUCED):
            ax = axes[row, ci]
            ax.plot(real_phys[ci], color='steelblue',    linewidth=1.5, label='real')
            ax.plot(sim_phys[ci],  color='tomato',       linewidth=1.5, linestyle='--', label='nearest sim')
            ax.set_title(f'{patients[pi]["file"]} — {ch}\ndist={dist_arr[pi]:.2f}', fontsize=8)
            ax.set_ylabel('mmHg', fontsize=7); ax.tick_params(labelsize=6)
            if row == 0 and ci == 0: ax.legend(fontsize=7)
    plt.suptitle(title, fontsize=11)
    plt.tight_layout(); plt.show()

sorted_idx = np.argsort(nn_dist_fft.numpy())
best3  = sorted_idx[:3].tolist()
worst3 = sorted_idx[-3:].tolist()

print('=== Best 3 matches (Fourier space) ===')
plot_overlay(best3,  nn_idx_fft[best3].tolist(),  'Fourier NN — best 3',  nn_dist_fft.numpy())

print('=== Worst 3 matches (Fourier space) ===')
plot_overlay(worst3, nn_idx_fft[worst3].tolist(), 'Fourier NN — worst 3', nn_dist_fft.numpy())

## MAF-joint posterior on nearest sim

Condition the base MAF-joint posterior on `x_sim[nn_idx]` — the 809-dim reduced waveform
of the nearest sim. The encoder inside the posterior (`enc_maf_joint`) maps it to the latent
space the flow was trained on. Retrieval-based: no domain adaptation.

In [ ]:
if CACHE_THETA.exists():
    t0 = time.time()
    theta_sim = torch.load(CACHE_THETA, map_location='cpu')
    print(f'theta_sim loaded in {time.time()-t0:.1f}s  shape: {tuple(theta_sim.shape)}')
else:
    print(f'Theta cache not found at {CACHE_THETA} — skipping nearest-sim param ceiling check.')
    theta_sim = None

In [ ]:
cas_idx = PARAM_KEYS_INFER.index('Cas')
eap_idx = PARAM_KEYS_INFER.index('Eap')
rap_idx = PARAM_KEYS_INFER.index('Rap')
ras_idx = PARAM_KEYS_INFER.index('Ras')

print(f'Sampling MAF-joint posterior on Fourier NN sim (N={N_POSTERIOR_SAMPLES}) — all {len(patients)} patients:\n')

cas_nn, eap_nn, rap_nn, ras_nn = [], [], [], []
theta_nn_means = []
t0 = time.time()

for pi, pat in enumerate(patients):
    x_nn = x_sim[nn_idx_fft[pi]].to(device)
    with torch.no_grad():
        samples = posterior.sample((N_POSTERIOR_SAMPLES,), x=x_nn, show_progress_bars=False)
    s_cpu = samples.cpu()
    in_prior = ((s_cpu >= lo_t) & (s_cpu <= hi_t)).all(dim=1).float().mean().item()
    means = s_cpu.mean(dim=0).numpy()
    theta_nn_means.append(means)
    cas_nn.append(means[cas_idx]); eap_nn.append(means[eap_idx])
    rap_nn.append(means[rap_idx]); ras_nn.append(means[ras_idx])
    print(f'  [{pi:2d}] nn_dist={nn_dist_fft[pi]:.3f}  '
          f'Cas={cas_nn[-1]:.3f} (GT={pat["gt"].get("Cas", float("nan")):.3f})  '
          f'Eap={eap_nn[-1]:.3f} (GT={pat["gt"].get("Eap", float("nan")):.3f})  '
          f'in_prior={in_prior:.2f}  ({time.time()-t0:.0f}s)')

cas_nn = np.array(cas_nn); eap_nn = np.array(eap_nn)
rap_nn = np.array(rap_nn); ras_nn = np.array(ras_nn)
theta_nn_means = np.stack(theta_nn_means)

cas_gt = np.array([p['gt'].get('Cas', np.nan) for p in patients])
eap_gt = np.array([p['gt'].get('Eap', np.nan) for p in patients])
rap_gt = np.array([p['gt'].get('PVR', np.nan) for p in patients])
ras_gt = np.array([p['gt'].get('SVR', np.nan) for p in patients])

mape_cas = np.nanmean(np.abs(cas_nn - cas_gt) / (np.abs(cas_gt) + 1e-9)) * 100
mape_eap = np.nanmean(np.abs(eap_nn - eap_gt) / (np.abs(eap_gt) + 1e-9)) * 100
mape_rap = np.nanmean(np.abs(rap_nn - rap_gt) / (np.abs(rap_gt) + 1e-9)) * 100
mape_ras = np.nanmean(np.abs(ras_nn - ras_gt) / (np.abs(ras_gt) + 1e-9)) * 100
print(f'\nMAPE  Cas={mape_cas:.1f}%  Eap={mape_eap:.1f}%  Rap(PVR)={mape_rap:.1f}%  Ras(SVR)={mape_ras:.1f}%')

## GT vs posterior mean — Cas, Eap, Rap (PVR), Ras (SVR)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, nn, gt, name, color in [
    (axes[0], cas_nn, cas_gt, 'Cas',       'steelblue'),
    (axes[1], eap_nn, eap_gt, 'Eap',       'tomato'),
    (axes[2], rap_nn, rap_gt, 'Rap (PVR)', 'mediumseagreen'),
    (axes[3], ras_nn, ras_gt, 'Ras (SVR)', 'orchid'),
]:
    valid = ~np.isnan(nn) & ~np.isnan(gt)
    lo = np.nanmin([gt[valid].min(), nn[valid].min()])
    hi = np.nanmax([gt[valid].max(), nn[valid].max()])
    ax.scatter(gt[valid], nn[valid], s=35, alpha=0.8, color=color, marker='^', zorder=3)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
    ax.set_xlabel(f'{name} measured'); ax.set_ylabel(f'{name} NN posterior mean')
    mape = np.nanmean(np.abs(nn[valid] - gt[valid]) / (np.abs(gt[valid]) + 1e-9)) * 100
    ax.set_title(f'{name}  MAPE={mape:.1f}%  (n={valid.sum()})')
fig.suptitle(f'GT vs NN posterior mean — Fourier KNN, {BASE_RUN}', fontsize=11)
plt.tight_layout(); plt.show()

## MAPE vs NN distance

Does a smaller Fourier distance predict better posterior accuracy?

In [ ]:
cas_ape = np.abs(cas_nn - cas_gt) / (np.abs(cas_gt) + 1e-9) * 100
eap_ape = np.abs(eap_nn - eap_gt) / (np.abs(eap_gt) + 1e-9) * 100
rap_ape = np.abs(rap_nn - rap_gt) / (np.abs(rap_gt) + 1e-9) * 100
ras_ape = np.abs(ras_nn - ras_gt) / (np.abs(ras_gt) + 1e-9) * 100
nn_dist_np = nn_dist_fft.numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, ape, name, color in [
    (axes[0], cas_ape, 'Cas', 'steelblue'),
    (axes[1], eap_ape, 'Eap', 'tomato'),
    (axes[2], rap_ape, 'Rap (PVR)', 'mediumseagreen'),
    (axes[3], ras_ape, 'Ras (SVR)', 'orchid'),
]:
    ax.scatter(nn_dist_np, ape, s=45, alpha=0.8, color=color, zorder=3)
    z_fit  = np.polyfit(nn_dist_np, ape, 1)
    x_line = np.linspace(nn_dist_np.min(), nn_dist_np.max(), 100)
    ax.plot(x_line, np.polyval(z_fit, x_line), 'k--', linewidth=1.2,
            label=f'slope={z_fit[0]:.1f}')
    ax.set_xlabel('NN distance (Fourier L2)')
    ax.set_ylabel('Absolute percentage error (%)')
    ax.set_title(f'{name}  MAPE={np.nanmean(ape):.1f}%')
    ax.legend(fontsize=8)
fig.suptitle('Error vs NN distance — does Fourier proximity predict accuracy?', fontsize=11)
plt.tight_layout(); plt.show()

## Nearest-sim parameters vs GT (ceiling check)

How close are the retrieved sim's own parameters to the patient's known GT?
The posterior should at most match the nearest sim's params.

In [ ]:
if theta_sim is not None:
    theta_nn = theta_sim[nn_idx_fft]   # (60, 25)

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, idx, gt, name, color in [
        (axes[0], cas_idx, cas_gt, 'Cas',       'steelblue'),
        (axes[1], eap_idx, eap_gt, 'Eap',       'tomato'),
        (axes[2], rap_idx, rap_gt, 'Rap (PVR)', 'mediumseagreen'),
        (axes[3], ras_idx, ras_gt, 'Ras (SVR)', 'orchid'),
    ]:
        nn_param = theta_nn[:, idx].numpy()
        valid = ~np.isnan(gt)
        lo = np.nanmin([gt[valid].min(), nn_param[valid].min()])
        hi = np.nanmax([gt[valid].max(), nn_param[valid].max()])
        ax.scatter(gt[valid], nn_param[valid], s=35, alpha=0.8, color=color, marker='s', zorder=3)
        ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
        ax.set_xlabel(f'{name} measured'); ax.set_ylabel(f'{name} nearest sim θ')
        mape = np.nanmean(np.abs(nn_param[valid] - gt[valid]) / (np.abs(gt[valid]) + 1e-9)) * 100
        ax.set_title(f'{name}  MAPE={mape:.1f}%  (n={valid.sum()})')
    fig.suptitle('GT vs nearest-sim θ — ceiling check (Fourier KNN)', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('theta_sim not loaded — skipping ceiling check')

## Posterior distributions — all params, first 4 patients

In [ ]:
N_PLOT   = min(4, len(patients))
N_PARAMS = len(PARAM_KEYS_INFER)

for pi, pat in enumerate(patients[:N_PLOT]):
    x_nn = x_sim[nn_idx_fft[pi]].to(device)
    with torch.no_grad():
        s = posterior.sample((N_POSTERIOR_SAMPLES,), x=x_nn, show_progress_bars=False).cpu().numpy()

    ncols, nrows = 6, 5
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 12))
    for i, name in enumerate(PARAM_KEYS_INFER):
        ax = axes[i // ncols, i % ncols]
        ax.hist(s[:, i], bins=40, alpha=0.7, color='mediumseagreen', density=True)
        gt_val = pat['gt'].get(name)
        if gt_val is not None:
            ax.axvline(gt_val, color='black', linewidth=1.5, linestyle='--', label='GT')
        ax.set_title(name, fontsize=8); ax.tick_params(labelsize=6)
        if i == 0: ax.legend(fontsize=7)
    for i in range(N_PARAMS, nrows * ncols):
        axes[i // ncols, i % ncols].axis('off')
    fig.suptitle(
        f'Posterior (Fourier NN) — {pat["file"]}  nn_dist={nn_dist_fft[pi]:.3f}  ({BASE_RUN})',
        fontsize=11
    )
    plt.tight_layout(); plt.show()

## Comparison table

Fourier-space KNN vs latent-space KNN (from OT eval notebooks).
Key question: does the learned OT encoder improve NN retrieval over raw Fourier geometry?

In [ ]:
print('=== MAPE comparison: retrieval methods ===')
print(f'{"Method":<55} {"Cas":>8} {"Eap":>8} {"Rap":>8} {"Ras":>8}')
print('-' * 91)
print(f'{"Fourier-KNN (unweighted, base MAF-joint)":<55} '
      f'{mape_cas:>7.1f}% {mape_eap:>7.1f}% {mape_rap:>7.1f}% {mape_ras:>7.1f}%')

# Reference values from OT eval notebooks (fill in after running)
print()
print('Reference MAPE from OT eval notebooks (latent-space KNN):')
print(f'{"Latent-KNN (enc_ot, OT-sinkhorn v2)":<55} {"28.9%":>8} {"177.4%":>8} {"128.9%":>8} {"23.6%":>8}')
print()
print('If Fourier-KNN MAPE is similar to or better than latent-KNN:')
print('  → the learned encoder adds limited value for retrieval')
print('  → Fourier-space retrieval is a strong baseline')